# P2 inference map and T4 benchmark

Notebook này chỉ load checkpoint đã chọn và dataset manifest của Protocol A
hoặc B. Nó không train, không resume optimizer và không đọc label raster.

Benchmark và map output dùng thư mục riêng, được khóa bằng SHA-256 của manifest
và checkpoint.

## 1. Mount Drive và checkout mã nguồn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hongphuc-maker/xuan-thuy-prithvi-segmentation.git'
REPO_REF = 'main'  # Replace with an approved commit SHA before a scientific run.
REPO_ROOT = Path('/content/xuan-thuy-prithvi-segmentation')

if not (REPO_ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_ROOT)])
else:
    dirty = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('The Colab repository copy has local changes; use a fresh runtime or review them.')

subprocess.check_call(['git', '-C', str(REPO_ROOT), 'fetch', '--depth=1', 'origin', REPO_REF])
subprocess.check_call(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', 'FETCH_HEAD'])

REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', REPO_COMMIT)


## 2. Cài inference environment

In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'terratorch==1.2.12', 'torchgeo==0.9.0', 'timm==1.0.30',
    'segmentation-models-pytorch==0.5.0', 'huggingface-hub==1.33.0',
    'einops==0.8.2', 'rasterio==1.5.2', 'pandas==2.2.3',
    'matplotlib>=3.8,<4', 'tqdm>=4.66,<5',
    'git+https://github.com/hongphuc-maker/xuan-thuy-dmi-segmentation.git@0775770ae87b83b7d2982ed727226128685f3fba'
])

import sys
sys.path.insert(0, str(REPO_ROOT / 'reference' / 'wce_completed'))
sys.path.insert(0, str(REPO_ROOT))


## 3. Chọn dataset manifest, checkpoint và output riêng

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Prithvi_Colab')
MANIFEST_PATH = DRIVE_ROOT / 'data' / 'inference' / 'REPLACE' / 'inference_dataset.json'
CHECKPOINT_PATH = DRIVE_ROOT / 'runs' / 'P2_multitemporal6_sep21_UNet_WCE' / 'best_any.pt'
BASE_CONFIG_PATH = REPO_ROOT / 'configs' / 'experiments' / 'p2_prithvi_unet_wce_completed.json'

# Use a new directory for every dataset/checkpoint pair. The inference code seals it.
OUTPUT_DIR = DRIVE_ROOT / 'inference_runs' / 'REPLACE_PROTOCOL' / 'REPLACE_DATASET' / 'REPLACE_CHECKPOINT_HASH_PREFIX'

assert MANIFEST_PATH.is_file(), MANIFEST_PATH
assert CHECKPOINT_PATH.is_file(), CHECKPOINT_PATH
assert BASE_CONFIG_PATH.is_file(), BASE_CONFIG_PATH
print(MANIFEST_PATH, CHECKPOINT_PATH, OUTPUT_DIR, sep='\n')


## 4. Audit dataset và load checkpoint P2

In [ ]:
import torch
from IPython.display import display
from deployment.inference_pipeline import InferenceScene, load_wce_model

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU')

scene = InferenceScene(MANIFEST_PATH, verify=True)
model, inference_config, checkpoint_provenance, device = load_wce_model(
    BASE_CONFIG_PATH, scene.manifest, CHECKPOINT_PATH
)
display(scene.audit)
display(checkpoint_provenance)
print('CUDA device:', torch.cuda.get_device_name(device))


## 5. T4 benchmark cell

In [ ]:
import json
from deployment.inference_pipeline import benchmark_inference

REQUIRE_T4 = True
device_name = torch.cuda.get_device_name(device)
if REQUIRE_T4 and 'T4' not in device_name.upper():
    raise RuntimeError(f'This benchmark is reserved for a T4 runtime; active device is {device_name}')

benchmark = benchmark_inference(model, scene, device, sample_tiles=30, warmup_tiles=5)
display(benchmark)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
benchmark_path = OUTPUT_DIR / 'T4_BENCHMARK.json'
benchmark_path.write_text(json.dumps(benchmark, indent=2), encoding='utf-8')
print('Saved:', benchmark_path)


## 6. Tạo map inference

`RUN_MAP` mặc định tắt. Khi bật, output được ghi cùng class, confidence,
entropy, common-validity, lock và measured full-map wall time. Không diễn giải
F1/IoU nếu chưa có ground truth độc lập đúng thời gian.

In [ ]:
from deployment.inference_pipeline import export_probability_map

RUN_MAP = False
if not RUN_MAP:
    print('Map inference is disabled. Review the audit and benchmark before setting RUN_MAP=True.')
else:
    result = export_probability_map(
        model, scene, checkpoint_provenance, OUTPUT_DIR, device, progress=True
    )
    display(result)


## 7. Preview output đã hoàn tất

In [ ]:
import json
import matplotlib.pyplot as plt
import rasterio

marker = OUTPUT_DIR / 'INFERENCE_COMPLETE.json'
if not marker.exists():
    print('No completed map in this output directory.')
else:
    result = json.loads(marker.read_text())
    class_file = next(name for name in result['artifacts'] if name.startswith('classes_'))
    with rasterio.open(OUTPUT_DIR / class_file) as source:
        classes = source.read(1)
    plt.figure(figsize=(12, 8))
    plt.imshow(classes, vmin=0, vmax=11, cmap='tab20')
    plt.title(f"{result['protocol']} | {result['dataset_id']} | {result['target_date']}")
    plt.axis('off')
    plt.show()
    display(result)
